## **Règles de nettoyage**

### **1. raw_campaign_spend_export**

Nous avons 3 cas différents.

**CAS A : doublon exact**  

Si toutes les colonnes sont identiques, on pourra garder une seule ligne

**CAS B : même nom de campagne + même période (debut et fin), mais données différentes**

Par exemple :   
campaign_name : promo ramadan bissap  
date_start : 2026-04-01  
date_end : 2026-04-29

Mais : 
objective : CONVERSIONS  
impressions : 53 488  
clicks : 834 clicks

objective : TRAFFIC  
impressions : 226 620  
clicks : 2 565 clicks  

Ce ne sont clairement pas des doublons exacts : les métriques et l'objectif diffèrent.  
Donc on supprimera uniquement lorsque toutes les valeurs des colonnes sont identiques





#### **Normalisation des objectifs**

Nous avons actuellement :

- Trafic 
- Traffic
- trafic
- Engagement
- engagement
- CONVERSIONS
- Conversions
- Notoriété
- Notoriete
- awareness
- conv

Nous allons créer quelque chose comme :  

Engagement / engagement -> engagement  
Notoriété / Notoriete / awareness -> awareness  
CONVERSIONS / Conversions / conv -> conversions  

Mais nous conserverons également l'original dans une colonne du type objective_raw et objective_normalized pour la traçabilité

#### **Dépenses**

Nous avons des formats comme :
- 1 100 000 FCFA
- 114 900 FCFA
- 617700
- 68 900

Il faudra donc transformer spend en spend_fcfa DOUBLE tout en concervant les valeurs de départ (spend_raw)

#### **Impressions / clicks**

Les 6 NULL restent des NULL. On ne fait surtout pas :  
NULL -> 0 car cela changerait le sens de la donnée. Par exemple, il eest normale que Radio et Influenceur n'aient pas de clicks et d'impressions

### **2. raw_media_plan**

#### **Channel**

On va créer :
- FB/IG -> Meta
- Google -> Google Ads ...

#### **Facturation**

On conserve invoiced_fcfa = NULL lorsque l'information manque et on crée : invoiced_missing = TRUE/FALSE  
Les écarts planned vs invoiced restent visibles. Ils feront partie de notre contrôle qualité.

### **3. raw_pos_sales_daily**

#### **Dates manquantes**

Nous avons exactement 14 jours consécutifs sans données (13 -> 26 avril 2026)  
On ne va pas créer artificiellement des ventes à zéro. Dans le staging, nous garderons la période absente et nous pourrons créer un indicateur de qualité **is_missing_sales_day**

Ainsi, quand on calculera les taux d'évolution, nous pourrons éviter de comparer aveuglément une période complète à une période incomplète.

#### **Retour**
on a : 
- 58 lignes avec revenue_fcfa < 0 (-159 829 FCFA)
- 58 lignes avec units_sold < 0 (-120 unités)

On constate aussi qu'il y a des transactions négatives, synonyme d'un retour, nous allons donc conserver les lignes et dériver :
- gross_sales_revenue
- return_revenue
- net_revenue

Par exemple :

- revenue_fcfa = 30 000, on a vente brute.   
- revenue_fcfa = -2 500, on a retour. 

Le net sera ensuite :
- net_revenue = gross_sales + returns, où les retours restent négatifs.

#### **Communes**

Nous avons :
- Marcory
- MARCORY
- Yopougon
- YOPOUGON

Donc on aura deux nouvelle variable, commune_raw pour l'ancien valeur et commune_normalized pour la nouvelle valeur et on normalise en casse cohérente.

### **4. raw_whatsapp_orders**

Ici nous avons découvert quelque chose de très important.  
order_ref n'est pas une clé fiable   : 

Exemple :

WA-2601-0262 apparaît deux fois avec :

- deux dates différentes (2026-01-01 18:52 et 2026-01-24 14:05) ;
- deux téléphones différents (01 86 23 10 63 et 01 15 45 92 51);
- deux montants différents (1600.0 et 8200.0);
- deux statuts différents (annulé et livrée).
- deux lieux de livraison différentes (riviera3 et angre 8eme tranche)

Donc, on ne va pas faire drop_duplicates(order_ref) mais nous allons plutôt créer un identifiant technique (whatsapp_row_id) et conserver order_ref comme référence métier non fiable.  
Ensuite, éventuellement une variable order_ref_repeated = TRUE pour signaler les références réutilisées.

#### **Statuts**

Nous allons normaliser :
- livré, Livré, LIVRE, livrée vers delivered
- annulé, annulée vers cancelled
- en cours vers pending

#### **Montants**

Pour le moment les 131 valeurs manquantes resteront manquantes

#### **Téléphones**

On créera phone_raw et customer_phone_normalized, par exemple :
- 05 48 68 72 57 -> +2250548687257

### **5. raw_social_comments**

Ici les 25 comment_id répétés sont effectivement identiques dans les exemples C000397 et "Abonne toi à ma page ❤️" et présent deux fois à l'identique.  
Donc ici, un doublon exact pourra être supprimé mais nous allons conserver :
- comment_id
- platform
- post_id
- published_at
- author_handle	comment_text
- like_count
- reply_to_id

puis ajouter après l'IA pour :
- trouver le language (language)
- analyser le sentiment du client (sentiment)
- trouver le thème de la discussion (theme)
- trouver le produit dont le commentaire parle (product)
- Savoir si le commentaire est un spam ou nom (is_spam)

## **Décisions et constats complémentaires (audit du projet)**

Ces points ont été établis en recalculant les chiffres directement depuis le fichier Excel brut, puis en les comparant aux tables dbt. Chaque décision indique sa **conséquence sur les chiffres du client** et ce qui reste **à confirmer**.

### **6. `raw_campaign_spend_export` — dépenses en euros**

**Constat :** 4 lignes de dépense sont saisies en euros (`524,42 EUR`, `305,97 EUR`, `423,81 EUR` deux fois). Après déduplication des doublons exacts (voir cas A), il reste **3 lignes EUR**, toutes sur Google.

**Règle retenue :** conversion au taux fixe de **655,0 FCFA / EUR**, paramétré dans `dbt/dbt_project.yml` (`vars: eur_to_fcfa_rate`) et non écrit en dur dans le SQL. Le montant d'origine (`spend_raw`), la devise (`spend_currency`), le taux appliqué (`fx_rate_to_fcfa`) et le drapeau `converted_from_eur` sont conservés dans `stg_campaign_spend`.

**Conséquence :** les lignes EUR pèsent 821 501 FCFA, soit **5,2 %** de la dépense totale (15 842 101 FCFA) et **46 %** de la dépense Google. La parité officielle est 655,957 : l'utiliser ajouterait environ 1 200 FCFA au total, un écart négligeable.

**À confirmer :** le taux réellement facturé par la régie, si Kômian dispose de la facture.

### **7. `raw_pos_sales_daily` — lignes identiques (96) : conservées**

**Constat :** 96 lignes sont des copies exactes d'une autre ligne (même date, POS, SKU, unités, montant). **Toutes** appartiennent à `POS999` (« Entrepôt Marcory », canaux delivery et e-commerce). Hors `POS999`, aucune ligne n'est dupliquée.

**Décision : ne pas dédupliquer.** `POS999` regroupe de nombreuses commandes par jour ; deux commandes identiques le même jour (par exemple 6 bissap 1L à 9 000 FCFA) sont plausibles. Dédupliquer risquerait de supprimer de vraies ventes.

**Conséquence :** si ces lignes étaient en réalité des doublons d'export, le CA net serait surévalué de **440 600 FCFA (0,50 %)**, au plus 0,9 % selon le mois.

**À confirmer :** auprès du distributeur, si l'export de `POS999` contient une ligne par commande ou une ligne par jour et SKU.

**Autre constat :** hors `POS999`, 58 clés (date, POS, SKU) ont deux lignes. Chacune est une vente accompagnée de son retour : ce ne sont pas des doublons.

### **8. `raw_pos_sales_daily` — un même magasin sous plusieurs `pos_id`**

**Constat :** `pos_id` et `pos_name` sont en correspondance 1:1 (45 identifiants, 45 noms), mais **4 magasins changent d'identifiant et de libellé le 1er mai 2026** : le nom passe en majuscules et sans accents, et les deux identifiants ne vendent jamais le même jour.

| Magasin | Avant le 1er mai | À partir du 1er mai |
|---|---|---|
| Kiosque Yopougon Ananeraie | `POS014` | `POS103` |
| Maquis Angré Djorobité | `POS008` | `POS102` |
| Maquis Marcory Anoumabo | `POS023` | `POS105` |
| Station Treichville Habitat | `POS035` | `POS108` |

**Règle retenue :** clé canonique `pos_key` = nom sans accents, en minuscules, espaces normalisés (`stg_pos_sales_daily`), et dimension `int_pos_dimension` (1 ligne = 1 magasin). Normalisation **exacte uniquement**, aucune fusion approximative. Résultat : **45 identifiants → 41 magasins**. Un test dbt (`int_pos_dimension_ids`) vérifie qu'un `pos_id` n'appartient qu'à un magasin et qu'un magasin ne vend pas sous deux identifiants le même jour.

**Conséquence :** aucun effet sur le CA total ni sur les comptes mensuels (les deux identifiants ne coexistent jamais dans un mois), mais l'historique par magasin est maintenant continu : sans cette clé, un magasin semblait fermer fin avril et un « nouveau » apparaître en mai.

**Cas ambigus, NON fusionnés, à valider par Kômian :**
- `Boutique Treichville Avenue 16` (`POS032`, jusqu'au 29 avril) et `Boutique Treichville Avenue 16 (nouveau)` (`POS107`, dès le 1er mai) : très probablement le même magasin rebaptisé. Si oui, on passe à **40 magasins**, ce qui correspond aux « environ quarante points de vente » du brief.
- `Supermarché Yopougon Maroc` (`POS016`) et `Supermarché Yopougon Selmer` (`POS011`) : noms proches mais tous deux actifs sur toute la période, ce sont deux magasins distincts.

### **9. `raw_whatsapp_orders` — montants invraisemblables (23 commandes)**

**Constat :** sur 935 montants renseignés, la médiane est de **6 050 FCFA** et le plus grand montant plausible est **12 600 FCFA**. Mais **23 commandes** (2,5 %) ont un montant compris entre **1 400 000 et 11 550 000 FCFA**, par exemple 11 550 000 FCFA pour « 6 bissap 1l + 3 gingembre 33cl ». Il n'existe aucune valeur entre 12 600 et 1 400 000 : l'écart est de facteur 111. Ces 23 montants représentent **95,8 %** de la somme des montants.

**Décision :** ne rien corriger et ne rien supprimer. Le montant brut reste dans `amount_fcfa`. Un montant au-dessus de **100 000 FCFA** (paramètre `whatsapp_max_plausible_amount_fcfa`) est signalé `amount_outlier` et **exclu des totaux** (`amount_plausible_fcfa`). Les commandes elles-mêmes restent comptées.

**Hypothèse, non appliquée :** divisés par 1 000, ces montants tombent entre 1 400 et 11 550 FCFA, ce qui est plausible pour 2 à 6 bouteilles. Une erreur d'unité (×1 000) est probable, mais elle doit être confirmée avant toute correction.

**Conséquence sur les chiffres :**

| | Avant | Après |
|---|---|---|
| Somme des montants connus (`known_order_amount_fcfa`) | 128 196 700 FCFA | **5 396 700 FCFA** |
| Montants exclus (tracés dans `outlier_amount_fcfa`) | — | 122 800 000 FCFA (23 commandes) |

**Effet sur le revenu livraison :** sur 624 commandes livrées, 82 n'ont pas de montant et 14 ont un montant aberrant : **96 commandes (15,4 %) n'ont pas de montant exploitable**. Le montant connu et plausible des commandes livrées est de 3 135 400 FCFA (`delivered_known_amount_fcfa`). **Aucun revenu livraison total n'est estimé** : ce serait extrapoler à partir de données absentes ou douteuses.

**À confirmer :** l'unité des montants WhatsApp (au franc ou au millier).

### **10. `raw_whatsapp_orders` — quantités écrites « Nx SKU »**

**Constat :** les textes du type `6x BIS-1L` étaient lus avec une quantité de **1**. 74 lignes sont concernées ; 64 avaient une quantité fausse (les 10 autres valaient bien 1).

**Correction :** le multiplicateur en tête (`6x`, `6 x`) est maintenant reconnu (`int_whatsapp_items`), et un test dbt (`int_whatsapp_items_multiplier`) compare la quantité extraite au multiplicateur écrit dans le texte.

**Conséquence :** unités WhatsApp **4 203 → 4 399** (+196, soit +4,7 %). Le parsing des autres formats a été vérifié : sur 441 commandes à une seule ligne et 377 commandes multi-lignes, aucune quantité ne diffère du texte.

### **11. Taux de réachat livraison — redéfini sur les commandes livrées**

**Constat :** le taux de 74,3 % (289 clients sur 389) comptait aussi les commandes **annulées** (296) et **en cours** (146). Un client dont les deux commandes ont été annulées n'est pas un client qui a racheté.

**Définition retenue :** taux de réachat = clients avec **au moins 2 commandes livrées** / clients avec **au moins 1 commande livrée**, soit **180 / 313 = 57,5 %**. L'ancienne définition reste disponible (`is_repeat_customer_any_status`) pour comparaison.

**Limite inchangée :** `customer_key` est un téléphone normalisé, pas une identité vérifiée.

### **12. Ventes — jours manquants et mois partiels**

Le calendrier des ventes (`int_sales_calendar`) est désormais **déduit des données** : du premier jour du premier mois au dernier jour du dernier mois observés. Il n'est plus écrit en dur sur janvier–juin 2026.

Conséquence : quand un mois de ventes arrive incomplet, ses jours absents sont comptés comme manquants et le dashboard affiche l'avertissement. Vérifié par simulation : 7 jours de ventes reçus pour juillet donnent 31 jours calendaires, 24 jours manquants. Avant la correction, ce mois s'affichait sans jours manquants, donc comme un mois complet.

Le test `int_sales_calendar_missing_period` n'impose plus la panne d'avril 2026 : il vérifie qu'un jour marqué manquant n'a aucune vente, et qu'un jour non marqué en a au moins une. Une nouvelle panne est donc **signalée**, elle ne fait pas échouer le run.

### **13. Produits des ventes POS — dimension et SKU inconnus**

**Constat :** `product_sku` est propre : 5 codes (`BIS-1L`, `BIS-33`, `BOU-1L`, `GIN-1L`, `GIN-33`), sans variante d'écriture.

**Règle retenue :** `int_product_dimension` traduit chaque SKU en produit (`BIS` → bissap, `GIN` → gingembre, `BOU` → bouye), format (`1L`, `33cl`) et volume en litres (1,0 ou 0,33). Seuls les préfixes et formats **connus** sont traduits : un SKU inconnu garde `product` et `format` à NULL, et le test `int_product_dimension_mapped` échoue pour qu'on complète le mapping. On n'invente jamais un produit ni un format, et un produit ne disparaît pas silencieusement du mix.

**Réconciliation :** `mart_product_mix_monthly` (mois × SKU) se réconcilie exactement avec `mart_sales_monthly` (87 646 311 FCFA et 75 945 unités nets), vérifié mois par mois par le test `mart_product_mix_monthly_reconciles`.

**Prix :** aucun prix n'est écrit en dur. Le CA net par litre observé est de 1 478 à 1 682 FCFA selon le produit.

Nous sommes maintenant prêt à passer au staging